# Práctica: clasificando reseñas de compradores

En `reseñas.csv` tienes 20 reseñas de un e-commerce, cada una con su etiqueta de sentimiento (`positiva` o `negativa`). Es un dataset chico a propósito: no se trata de lograr el mejor accuracy posible, sino de que construyas el pipeline completo con tus propias manos y entiendas qué hace cada paso.
 
Antes de escribir código, leé las 20 reseñas del CSV a mano. Fijate qué palabras aparecen repetidas en las positivas y cuáles en las negativas. Esa intuición te va a servir después para saber si el modelo está aprendiendo algo razonable o si está fallando por algo que vos ya podías anticipar.

## Ejercicio 1: cargar y explorar
 
Cargá el CSV con Pandas. Contá cuántas reseñas hay de cada clase (`positiva` / `negativa`). Esto no es un paso decorativo: si las clases estuvieran muy desbalanceadas (por ejemplo 18 positivas y 2 negativas), un modelo podría lograr un accuracy alto simplemente prediciendo siempre "positiva", sin haber aprendido nada útil. Antes de entrenar cualquier modelo de clasificación, siempre chequeá el balance de clases.

In [11]:
# tu codigo acá
import pandas as pd
df = pd.read_csv("/workspaces/data-analysis-course/modulo-4-visualizacion/clase-15/practica/reseñas.csv")

negativas = (df['sentimiento'] == 'negativa').sum()
positivas = (df['sentimiento'] == 'positiva').sum()

print("\nResultados en variables:")
print(f"Negativas: {negativas}")
print(f"Positivas: {positivas}")


Resultados en variables:
Negativas: 10
Positivas: 10


In [12]:
print(df)

                                                texto sentimiento
0   el producto llegó en perfecto estado y antes d...    positiva
1   pésima experiencia, el paquete llegó abierto y...    negativa
2   muy buena relación precio calidad, lo volvería...    positiva
3   el vendedor nunca respondió mis mensajes, mala...    negativa
4          superó mis expectativas, calidad excelente    positiva
5      se rompió a la semana de uso, no lo recomiendo    negativa
6           envío rapidísimo, tal cual la descripción    positiva
7   la caja llegó totalmente destruida y el produc...    negativa
8   funciona perfecto, instalación sencilla y buen...    positiva
9     tardó un mes en llegar y encima vino incompleto    negativa
10  excelente atención del vendedor, resolvió toda...    positiva
11  la calidad es muy inferior a lo que mostraban ...    negativa
12           quedé conforme, cumplió con lo prometido    positiva
13      no funciona como se anuncia, una estafa total    negativa
14     bue

## Ejercicio 2: tokenizar sin librerías
 
Escribí una función `tokenizar_simple(texto)` que reciba un string y devuelva una lista de palabras en minúscula, sin signos de puntuación, usando solo métodos nativos de Python (sin NLTK ni spaCy). Pista: puede que necesites la librería `string` y su lista de puntuación, además de `.split()`.
 
Después, aplicá `word_tokenize` de NLTK sobre la misma reseña y compará los resultados. ¿En qué casos tu función simple se equivoca o produce un resultado distinto al de NLTK? Escribí al menos un ejemplo concreto de una reseña del dataset donde la diferencia se note.

In [24]:
import string
texto = "quedé conforme, cumplió con lo prometido"

def tokenizar_simple ():
   return texto.split()

tokens = tokenizar_simple()
print(tokens)

['quedé', 'conforme,', 'cumplió', 'con', 'lo', 'prometido']


In [37]:
pip install nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.8/801.8 kB 24.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]]]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [38]:
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

texto = "quedé conforme, cumplió con lo prometido"
tokens = word_tokenize(texto, language='spanish')
print(tokens)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


['quedé', 'conforme', ',', 'cumplió', 'con', 'lo', 'prometido']


## Ejercicio 3: el efecto de las stopwords
 
Tomá la reseña `"no lo recomiendo, una pérdida de dinero total"` (o equivalente del dataset) y sacale las stopwords en español con NLTK. Mirá el resultado.
 
Ahora respondé sin correr más código, solo pensando: si esta reseña fuera parte de un modelo de Bag of Words que sacó stopwords, y la palabra "no" desapareció, ¿qué información se perdió? Buscá en el dataset si hay alguna otra reseña donde sacar "no" cambiaría el sentido de la frase. Este ejercicio no tiene una única respuesta "correcta" en código: el objetivo es que argumentes cuándo sacar stopwords ayuda y cuándo perjudica.

In [41]:
from nltk.corpus import stopwords
nltk.download('stopwords')

texto = "no lo recomiendo, una pérdida de dinero total"
tokens = word_tokenize(texto.lower())

stop_words = set(stopwords.words('spanish'))
tokens_filtrados = [t for t in tokens if t.lower() not in stop_words]

print (tokens_filtrados)

['recomiendo', ',', 'pérdida', 'dinero', 'total']


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Ejercicio 4: vectorizar con TF-IDF
 
Usando `TfidfVectorizer` de sklearn, vectorizá las 20 reseñas del dataset completo (no hace falta separar en train/test todavía). Imprimí el vocabulario completo que generó el vectorizador con `.get_feature_names_out()`.
 
Buscá en ese vocabulario las 5 palabras con el IDF más alto (es decir, las más "raras" o distintivas del corpus) y las 5 con el IDF más bajo. Podés acceder a esos valores con el atributo `.idf_` del vectorizador, en el mismo orden que el vocabulario. ¿Las palabras con IDF alto te parecen informativas sobre el sentimiento de la reseña donde aparecen?

In [46]:
pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 20.0 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 42.3 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [60]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = df['texto'].tolist()

tfidf = TfidfVectorizer()
matriz_tfidf = tfidf.fit_transform(corpus)

print(tfidf.get_feature_names_out())
print(matriz_tfidf.toarray().round(2))

['abierto' 'antes' 'anuncia' 'atención' 'avisarme' 'bien' 'buen' 'buena'
 'buenísima' 'caja' 'calidad' 'cancelaron' 'coincide' 'color' 'como'
 'compra' 'comprar' 'con' 'conforme' 'cual' 'cumplió' 'dañado' 'de' 'del'
 'descripción' 'destruida' 'diez' 'dudas' 'el' 'empaquetado' 'en' 'encima'
 'envío' 'es' 'esperado' 'estado' 'estafa' 'excelente' 'expectativas'
 'experiencia' 'faltaban' 'familia' 'foto' 'fotos' 'funciona' 'incompleto'
 'inferior' 'instalación' 'justo' 'la' 'las' 'llegar' 'llegó' 'lo' 'mala'
 'material' 'mensajes' 'mes' 'mi' 'mis' 'mostraban' 'muy' 'nada' 'no'
 'nunca' 'paquete' 'para' 'pedido' 'pedí' 'perfecto' 'piezas' 'precio'
 'producto' 'prometido' 'publicada' 'pésima' 'que' 'quedé' 'rapidísimo'
 'recomendé' 'recomiendo' 'reembolso' 'relación' 'resolvió' 'respondió'
 'rompió' 'se' 'semana' 'sencilla' 'servicio' 'sin' 'superó' 'tal' 'tardó'
 'terrible' 'todas' 'total' 'totalmente' 'un' 'una' 'uso' 'vendedor'
 'vino' 'volvería' 'ya']
[[0.   0.38 0.   ... 0.   0.   0.  ]